# 📊 Linear Regression — Memprediksi Angka dengan Garis Terbaik

**Modul 1: Machine Learning Fundamentals** | Notebook 1 dari 9

---

## 🎯 Apa yang Akan Kita Pelajari?

Di notebook ini, kita akan belajar:
1. Apa itu **Linear Regression** dan kapan kita menggunakannya
2. Bagaimana cara **melatih model** untuk memprediksi angka
3. Bagaimana cara **mengevaluasi** apakah prediksi kita bagus atau tidak
4. Teknik **regularisasi** untuk membuat model lebih stabil *(materi bonus)*

### ⏱️ Estimasi Waktu: ~90 menit + diskusi

---

## 💡 Analogi Sederhana

Bayangkan kamu punya **toko es krim**. Kamu perhatikan bahwa:
- Kalau hari **panas**, es krim **laku banyak**
- Kalau hari **dingin**, es krim **laku sedikit**

Nah, kamu ingin **menebak** berapa es krim yang akan terjual besok berdasarkan ramalan cuaca.

**Linear Regression** = cara matematika untuk menemukan **garis tebakan terbaik** dari data yang sudah ada.

<img src="https://upload.wikimedia.org/wikipedia/commons/thumb/3/3a/Linear_regression.svg/400px-Linear_regression.svg.png" width="300">

Garis merah di atas adalah "garis terbaik" — letaknya diatur supaya **jarak ke semua titik data sekecil mungkin**.

---

## 1️⃣ Persiapan — Install & Import Library

Pertama, kita siapkan alat-alatnya. Seperti mau masak, kita harus siapkan peralatan dapur dulu.

| Library | Fungsi |
|---------|--------|
| `pandas` | Membaca dan mengolah data (seperti Excel di Python) |
| `numpy` | Perhitungan matematika |
| `matplotlib` & `seaborn` | Membuat grafik dan visualisasi |
| `scikit-learn` | Library utama untuk Machine Learning |

In [ ]:
# Import library yang dibutuhkan
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Library Machine Learning
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Pengaturan tampilan grafik
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

print('✅ Semua library berhasil di-import!')

---

## 2️⃣ Memuat Data

### 📋 Tentang Dataset

Kita akan menggunakan dataset **Advertising** (Iklan). Dataset ini berisi data dari **200 pasar** yang mencatat:

| Kolom | Keterangan | Satuan |
|-------|-----------|--------|
| `TV` | Budget iklan di TV | Ribuan dollar |
| `radio` | Budget iklan di Radio | Ribuan dollar |
| `newspaper` | Budget iklan di Koran | Ribuan dollar |
| `sales` | Jumlah penjualan produk | Ribuan unit |

**Pertanyaan kita:** Jika kita tahu budget iklan di TV, Radio, dan Koran, bisakah kita **memprediksi jumlah penjualan**?

### Upload File
Jalankan cell di bawah, lalu pilih file `Advertising.csv` dari komputer kamu.

> 💡 **Tanpa upload manual?** Dataset ini juga tersedia di repo GitHub dan bisa dimuat otomatis — lihat baris komentar *Alternatif* pada cell kode di bawah (`pd.read_csv(<raw_url>)`).

In [ ]:
# Upload file CSV ke Google Colab
from google.colab import files
uploaded = files.upload()


# === Alternatif tanpa upload manual: muat langsung dari GitHub repo ===
# import pandas as pd
# df = pd.read_csv('https://raw.githubusercontent.com/chmdznr/navasena-gen-ml-course/master/01_machine_learning_fundamentals/Advertising.csv')


In [ ]:
# Baca file CSV ke dalam DataFrame
df = pd.read_csv('Advertising.csv')

# Lihat 5 baris pertama
print(f'Dataset memiliki {df.shape[0]} baris dan {df.shape[1]} kolom')
df.head()

In [ ]:
# Hapus kolom index yang tidak diperlukan
# Kolom 'Unnamed: 0' hanya berisi nomor urut, tidak berguna untuk prediksi
if 'Unnamed: 0' in df.columns:
    df = df.drop(columns='Unnamed: 0')

print(f'Setelah dibersihkan: {df.shape[0]} baris, {df.shape[1]} kolom')
df.head()

---

## 3️⃣ Eksplorasi Data (EDA)

Sebelum membuat model, kita harus **mengenal data** kita dulu. Ini seperti seorang dokter yang memeriksa pasien sebelum memberikan obat.

### Statistik Dasar
Mari kita lihat ringkasan statistik dari dataset.

In [ ]:
# Statistik deskriptif
# count = jumlah data, mean = rata-rata, std = standar deviasi (sebaran data)
# min/max = nilai terkecil/terbesar, 25%/50%/75% = persentil
df.describe()

**Apa yang bisa kita baca?**
- Budget iklan TV paling besar (rata-rata ~147 ribu dollar), sedangkan radio dan koran lebih kecil
- Penjualan rata-rata sekitar 14 ribu unit
- Tidak ada data yang kosong (count = 200 untuk semua kolom)

### Cek Data Kosong (Missing Values)

In [ ]:
# Cek apakah ada data yang kosong
print('Jumlah data kosong per kolom:')
print(df.isnull().sum())
print(f'\n✅ Total data kosong: {df.isnull().sum().sum()}')

### 📈 Visualisasi: Hubungan Antar Variabel

Sekarang kita lihat — apakah ada **hubungan** antara budget iklan dengan penjualan?

Kalau titik-titiknya membentuk pola **naik dari kiri bawah ke kanan atas**, berarti ada hubungan positif (makin besar budget → makin besar sales).

In [ ]:
# Buat scatter plot untuk melihat hubungan tiap variabel dengan sales
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

for i, col in enumerate(['TV', 'radio', 'newspaper']):
    axes[i].scatter(df[col], df['sales'], alpha=0.6, color=['#2196F3', '#4CAF50', '#FF9800'][i])
    axes[i].set_xlabel(f'Budget Iklan {col} (ribu $)', fontsize=11)
    axes[i].set_ylabel('Penjualan (ribu unit)', fontsize=11)
    axes[i].set_title(f'{col} vs Sales', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

**Pengamatan:**
- **TV** → Hubungan **paling jelas**! Makin besar budget TV, penjualan cenderung naik 📈
- **Radio** → Ada hubungan positif, tapi lebih menyebar
- **Newspaper** → Hubungannya **lemah**, titik-titik sangat menyebar

### 🔢 Korelasi — Mengukur Kekuatan Hubungan

Korelasi adalah angka antara **-1 sampai +1** yang mengukur seberapa kuat hubungan dua variabel:
- **+1** = hubungan positif sempurna (naik bersama)
- **0** = tidak ada hubungan
- **-1** = hubungan negatif sempurna (satu naik, satu turun)

In [ ]:
# Hitung korelasi antar semua variabel
corr = df.corr()

# Tampilkan sebagai heatmap (peta warna)
plt.figure(figsize=(8, 6))
sns.heatmap(corr, annot=True, cmap='coolwarm', center=0,
            fmt='.2f', square=True, linewidths=1)
plt.title('Heatmap Korelasi Antar Variabel', fontsize=14, fontweight='bold')
plt.show()

print('\n📊 Korelasi dengan Penjualan (sales):')
print(corr['sales'].drop('sales').sort_values(ascending=False).to_string())

**Kesimpulan EDA:**
- TV memiliki korelasi **tertinggi** (~0.78) dengan sales — ini prediktor terkuat
- Radio juga cukup berpengaruh (~0.58)
- Newspaper punya korelasi **rendah** (~0.23) — mungkin tidak terlalu membantu prediksi

---

## 4️⃣ Mempersiapkan Data untuk Model

### Konsep: Fitur (X) dan Target (y)

Dalam Machine Learning, kita membagi data menjadi:
- **Fitur (X)** = informasi yang kita *gunakan* untuk menebak → Budget TV, Radio, Newspaper
- **Target (y)** = hal yang ingin kita *tebak/prediksi* → Sales (penjualan)

**Analogi:** Seperti ujian sekolah — X adalah soal yang dikasih, y adalah jawaban yang harus ditebak.

### Konsep: Train-Test Split

Kita bagi data menjadi 2 bagian:
- **Data latihan (train)** = 70% data → untuk melatih model (seperti latihan soal)
- **Data ujian (test)** = 30% data → untuk menguji model (seperti ujian sesungguhnya)

Kenapa harus dipisah? Karena kita mau tahu apakah model bisa menebak **data yang belum pernah dilihat**, bukan hanya menghapal data latihan.

In [ ]:
# Pisahkan fitur (X) dan target (y)
X = df.drop(columns='sales')   # Semua kolom kecuali 'sales'
y = df['sales']                 # Hanya kolom 'sales'

print(f'Fitur (X): {list(X.columns)}')
print(f'Target (y): sales')
print(f'Jumlah data: {len(X)}')

In [ ]:
# Bagi data menjadi train (70%) dan test (30%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.3,     # 30% untuk test
    random_state=42    # Agar hasil selalu sama setiap kali dijalankan
)

print(f'Data latihan: {X_train.shape[0]} baris')
print(f'Data ujian:   {X_test.shape[0]} baris')
print(f'Rasio:        {X_train.shape[0]/len(X)*100:.0f}% train, {X_test.shape[0]/len(X)*100:.0f}% test')

---

## 5️⃣ Melatih Model Linear Regression

### Bagaimana Linear Regression Bekerja?

Linear Regression mencari rumus garis terbaik:

$$\text{sales} = w_1 \times TV + w_2 \times radio + w_3 \times newspaper + b$$

Di mana:
- $w_1, w_2, w_3$ = **koefisien** (bobot) — seberapa besar pengaruh masing-masing faktor
- $b$ = **intercept** (titik awal) — penjualan dasar tanpa iklan sama sekali

Model akan **belajar dari data latihan** untuk menemukan nilai $w$ dan $b$ terbaik.

In [ ]:
# Buat dan latih model Linear Regression
model = LinearRegression()
model.fit(X_train, y_train)   # Latih model dengan data training

print('✅ Model berhasil dilatih!')
print(f'\nRumus yang ditemukan model:')
print(f'  sales = {model.coef_[0]:.4f} × TV + {model.coef_[1]:.4f} × radio + {model.coef_[2]:.4f} × newspaper + {model.intercept_:.4f}')

**Cara membaca koefisien:**
- Koefisien TV ≈ 0.046 → Setiap tambahan 1 ribu dollar di iklan TV, penjualan naik ~46 unit
- Koefisien radio ≈ 0.18 → Setiap tambahan 1 ribu dollar di radio, penjualan naik ~180 unit
- Koefisien newspaper ≈ 0.003 → Iklan koran **hampir tidak berpengaruh** pada penjualan!

💡 **Menarik!** Meskipun budget TV lebih besar, ternyata **radio lebih efisien** per dollar-nya.

---

## 6️⃣ Membuat Prediksi

Sekarang kita gunakan model untuk **memprediksi penjualan** pada data ujian (yang belum pernah dilihat model).

In [ ]:
# Prediksi menggunakan data test
y_pred = model.predict(X_test)

# Bandingkan prediksi vs kenyataan (5 contoh pertama)
hasil = pd.DataFrame({
    'Penjualan Asli': y_test.values[:10],
    'Prediksi Model': y_pred[:10].round(2),
    'Selisih': (y_test.values[:10] - y_pred[:10]).round(2)
})
print('Perbandingan Prediksi vs Kenyataan (10 data pertama):')
hasil

In [ ]:
# Visualisasi: Prediksi vs Kenyataan
plt.figure(figsize=(8, 8))
plt.scatter(y_test, y_pred, alpha=0.6, color='#2196F3', edgecolors='white', s=80)

# Garis diagonal (prediksi sempurna)
min_val = min(y_test.min(), y_pred.min())
max_val = max(y_test.max(), y_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], 'r--', linewidth=2, label='Prediksi Sempurna')

plt.xlabel('Penjualan Asli (ribu unit)', fontsize=12)
plt.ylabel('Prediksi Model (ribu unit)', fontsize=12)
plt.title('Seberapa Akurat Prediksi Kita?', fontsize=14, fontweight='bold')
plt.legend(fontsize=11)
plt.tight_layout()
plt.show()

print('💡 Semakin dekat titik-titik ke garis merah, semakin akurat prediksi model.')

---

## 7️⃣ Evaluasi Model

Bagaimana kita tahu apakah model kita **bagus** atau tidak? Kita gunakan beberapa metrik:

| Metrik | Penjelasan Sederhana | Nilai Ideal |
|--------|---------------------|-------------|
| **MAE** | Rata-rata kesalahan prediksi (dalam satuan asli) | Mendekati 0 |
| **MSE** | Seperti MAE, tapi kesalahan besar dihukum lebih berat | Mendekati 0 |
| **RMSE** | Akar dari MSE, jadi satuannya sama dengan data asli | Mendekati 0 |
| **R² Score** | Persentase variasi data yang bisa dijelaskan model | Mendekati 1 (100%) |

**Analogi R² Score:**
Bayangkan kamu mengerjakan ujian 100 soal. R² = 0.90 artinya model bisa "menjawab benar" 90% pola dalam data.

In [ ]:
# Hitung metrik evaluasi
mae  = mean_absolute_error(y_test, y_pred)
mse  = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2   = r2_score(y_test, y_pred)

print('📊 Hasil Evaluasi Model Linear Regression')
print('=' * 45)
print(f'  MAE  (Rata-rata Kesalahan)  : {mae:.3f} ribu unit')
print(f'  MSE  (Mean Squared Error)   : {mse:.3f}')
print(f'  RMSE (Root Mean Squared Err): {rmse:.3f} ribu unit')
print(f'  R²   (Skor Akurasi)         : {r2:.3f} ({r2*100:.1f}%)')
print('=' * 45)

if r2 > 0.8:
    print('\n✅ Model cukup bagus! R² di atas 80%')
elif r2 > 0.5:
    print('\n⚠️ Model lumayan, tapi masih bisa ditingkatkan.')
else:
    print('\n❌ Model kurang bagus, perlu perbaikan.')

---

## 8️⃣ Mengecek Asumsi Linear Regression

Linear Regression punya beberapa "syarat" agar hasilnya bisa dipercaya. Mari kita cek satu per satu.

### Asumsi 1: Residual (Sisa Kesalahan) Terdistribusi Normal

**Residual** = selisih antara nilai asli dan prediksi. Idealnya, kesalahan model harus tersebar secara acak membentuk "lonceng" (distribusi normal).

In [ ]:
# Hitung residual (kesalahan prediksi)
residual = y_test - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram residual
sns.histplot(residual, kde=True, ax=axes[0], color='#2196F3')
axes[0].axvline(x=0, color='red', linestyle='--', label='Nol (ideal)')
axes[0].set_xlabel('Residual (Kesalahan)', fontsize=11)
axes[0].set_ylabel('Frekuensi', fontsize=11)
axes[0].set_title('Distribusi Residual', fontsize=13, fontweight='bold')
axes[0].legend()

# Scatter plot residual vs prediksi
axes[1].scatter(y_pred, residual, alpha=0.6, color='#4CAF50', edgecolors='white', s=60)
axes[1].axhline(y=0, color='red', linestyle='--')
axes[1].set_xlabel('Nilai Prediksi', fontsize=11)
axes[1].set_ylabel('Residual (Kesalahan)', fontsize=11)
axes[1].set_title('Residual vs Prediksi', fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()

print(f'Rata-rata residual: {np.mean(residual):.4f} (idealnya mendekati 0)')

**Apa yang kita harapkan?**
- Histogram berbentuk lonceng dan berpusat di sekitar 0 ✅
- Scatter plot residual tersebar **acak** (tidak membentuk pola) ✅

Jika residual membentuk pola (misalnya melengkung), itu artinya hubungan datanya **bukan linear** dan kita butuh metode lain.

---

## 🌟 BONUS: Regularisasi

Kadang model terlalu "semangat" mengikuti data latihan sampai jadi **overfitting** (menghapal, bukan memahami). Regularisasi adalah cara untuk menghindari ini.

**Analogi:** Bayangkan kamu belajar untuk ujian:
- **Tanpa regularisasi** = menghapal semua soal latihan → kalau soal ujian beda, bingung
- **Dengan regularisasi** = memahami konsepnya → bisa menjawab soal baru

### Tiga Jenis Regularisasi:

| Metode | Cara Kerja | Analogi |
|--------|-----------|----------|
| **Lasso (L1)** | Bisa membuat koefisien jadi **nol** (membuang fitur tidak penting) | Seperti memilih hanya mata pelajaran yang relevan |
| **Ridge (L2)** | Mengecilkan koefisien tapi **tidak sampai nol** | Seperti mengurangi porsi belajar semua mata pelajaran secara merata |
| **ElasticNet** | Gabungan Lasso + Ridge | Kombinasi kedua strategi di atas |

In [ ]:
from sklearn.linear_model import Lasso, Ridge, ElasticNet

# Dictionary untuk menyimpan hasil
hasil_model = {'Linear Regression': r2}

# Lasso Regression (L1)
model_lasso = Lasso(alpha=0.1)
model_lasso.fit(X_train, y_train)
y_pred_lasso = model_lasso.predict(X_test)
hasil_model['Lasso (L1)'] = r2_score(y_test, y_pred_lasso)

# Ridge Regression (L2)
model_ridge = Ridge(alpha=1.0)
model_ridge.fit(X_train, y_train)
y_pred_ridge = model_ridge.predict(X_test)  # Menggunakan model_ridge, BUKAN model_lasso!
hasil_model['Ridge (L2)'] = r2_score(y_test, y_pred_ridge)

# ElasticNet (L1 + L2)
model_elastic = ElasticNet(alpha=0.1, l1_ratio=0.5)
model_elastic.fit(X_train, y_train)
y_pred_elastic = model_elastic.predict(X_test)
hasil_model['ElasticNet'] = r2_score(y_test, y_pred_elastic)

# Tampilkan perbandingan
print('📊 Perbandingan R² Score Semua Model')
print('=' * 40)
for nama, skor in hasil_model.items():
    bar = '█' * int(skor * 30)
    print(f'  {nama:25s}: {skor:.4f} {bar}')
print('=' * 40)

In [ ]:
# Bandingkan koefisien setiap model
koef_df = pd.DataFrame({
    'Fitur': X.columns,
    'Linear Regression': model.coef_,
    'Lasso': model_lasso.coef_,
    'Ridge': model_ridge.coef_,
    'ElasticNet': model_elastic.coef_
})

print('📋 Perbandingan Koefisien (Bobot) Setiap Model:')
print('Perhatikan: Lasso cenderung membuat koefisien kecil jadi NOL')
print()
koef_df

---

## 📝 Ringkasan: Apa yang Kita Pelajari Hari Ini?

### Konsep Utama

1. **Linear Regression** mencari garis/bidang terbaik untuk memprediksi angka berdasarkan data yang ada
2. Kita membagi data menjadi **train** (latihan) dan **test** (ujian) agar bisa menguji kemampuan model pada data baru
3. **R² Score** menunjukkan seberapa bagus model kita (makin dekat ke 1.0 = makin bagus)
4. **Regularisasi** (Lasso, Ridge, ElasticNet) membantu mencegah model dari overfitting

### Temuan dari Data Advertising

- Iklan di **TV** dan **Radio** berpengaruh besar terhadap penjualan
- Iklan di **Koran** hampir tidak berpengaruh (koefisiennya mendekati nol)
- Model Linear Regression sudah cukup baik untuk dataset ini (R² > 90%)

### Kapan Menggunakan Linear Regression?

✅ **Gunakan jika:**
- Target prediksi berupa **angka kontinu** (harga, suhu, penjualan)
- Hubungan antara fitur dan target cukup **linear** (garis lurus)
- Data tidak terlalu kompleks

❌ **Jangan gunakan jika:**
- Target berupa **kategori** (ya/tidak, merah/biru) → gunakan Logistic Regression
- Hubungan datanya sangat **kompleks dan non-linear** → gunakan model lain (Decision Tree, dll)

---

### 🔜 Notebook Selanjutnya: Logistic Regression
Di notebook berikutnya, kita akan belajar memprediksi **kategori** (bukan angka), misalnya: apakah nasabah bank akan membuka deposito atau tidak?